# Band power — Continuous whole-block (Flow & Load papers)

Computes Welch PSD over entire task blocks from step7 preprocessed raw.
Works for both active and passive participants.

Band definitions (different from Castellar — do not mix):
- theta:     3.5–8Hz
- alpha:     8–13Hz
- low_beta:  13–20Hz
- high_beta: 20–30Hz

**Usage**: set `PAPER` and `EXPERIMENT` in Cell 1, then run all.

Last edit: 16.10.2026 @anguyen

In [ ]:
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.integrate import simpson

import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.io import read_raw_fif
from mne.viz import set_browser_backend

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from participants import get_participants

sns.set_theme(style='darkgrid')
%matplotlib qt
set_browser_backend('qt')

DATE_STR = datetime.today().strftime('%Y%m%d')

In [ ]:
# ==============================================================================
# CONFIGURATION — edit here
# ==============================================================================

PAPER      = 'load'    # 'flow' or 'load'
EXPERIMENT = 'passive'  # 'active' or 'passive'

# Band definitions — intentionally different from Castellar
OUR_BANDS = {
    'theta':     (3.5,  8.0),
    'alpha':     (8.0,  13.0),
    'low_beta':  (13.0, 20.0),
    'high_beta': (20.0, 30.0),
}

# group mapping for flow condition splitting
high_first = ['G2', 'G4', 'G6', 'G8']
low_first  = ['G1', 'G3', 'G5', 'G7']

# ==============================================================================
# FIXED — do not edit
# ==============================================================================

STUDY = 'Active' if EXPERIMENT == 'active' else 'Passive'

PARTICIPANTS      = get_participants(PAPER, EXPERIMENT)
participants_kept = list(PARTICIPANTS.keys())
n                 = len(participants_kept)
print(f'Loaded {n} participants ({PAPER} / {EXPERIMENT})')

if sys.platform == 'win32':
    path        = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')
    output_root = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs/measures/{PAPER}/freq/continuous')
else:
    path        = Path(f'/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')
    output_root = Path(f'/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2/outputs/measures/{PAPER}/freq/continuous')

output_root.mkdir(parents=True, exist_ok=True)

img_root = path / 'outputs' / 'img' / PAPER / 'freq' / 'continuous'
img_root.mkdir(parents=True, exist_ok=True)

print(f'Output: {output_root}')

In [ ]:
# ==============================================================================
# File discovery and condition splitting
# ==============================================================================

def extract_uid(file: str) -> str:
    m = re.search(r'sub-([A-Za-z0-9]+)-', file)
    if m is None:
        raise ValueError(f'Could not extract UID from: {file}')
    return m.group(1)


def gid_of(file: str) -> str | None:
    m = re.search(r'sub-[A-Za-z0-9]+-([A-Z0-9]+)', file)
    return m.group(1) if m else None


raw_files = [str(f) for f in path.rglob('*_step7_preprocessed_raw.fif')
             if any(p in str(f) for p in participants_kept)]
print(f'Raw files found: {len(raw_files)} (expected {n * 4})')

f_oddball_1   = [x for x in raw_files if 'oddball' in x and 'run-1' in x]
f_oddball_2   = [x for x in raw_files if 'oddball' in x and 'run-2' in x]
f_UT_highflow = ([x for x in raw_files if 'UT' in x and 'run-1' in x and gid_of(x) in high_first] +
                 [x for x in raw_files if 'UT' in x and 'run-2' in x and gid_of(x) in low_first])
f_UT_lowflow  = ([x for x in raw_files if 'UT' in x and 'run-1' in x and gid_of(x) in low_first] +
                 [x for x in raw_files if 'UT' in x and 'run-2' in x and gid_of(x) in high_first])

file_sets = {
    'oddball_1':   f_oddball_1,
    'oddball_2':   f_oddball_2,
    'UT_highflow': f_UT_highflow,
    'UT_lowflow':  f_UT_lowflow,
}
for cond, files in file_sets.items():
    print(f'  {cond}: {len(files)} files')

In [ ]:
# ==============================================================================
# Processing function — Welch PSD over entire block
# ==============================================================================

def process_file(file: str):
    participant = extract_uid(file)
    raw         = read_raw_fif(file, preload=True, verbose=False)
    raw.pick('eeg')
    
    # interpolate bad channels before PSD so all participants have full channel set
    if raw.info['bads']:
        raw.interpolate_bads(reset_bads=True, verbose=False)

    spectrum = raw.compute_psd(
        method='welch',
        fmin=1.0,
        fmax=40.0,
        n_fft=1024,
        n_per_seg=512,
        verbose=False,
        n_jobs=1,
    )
    df = spectrum.to_data_frame()
    df.insert(0, 'participant', participant)
    return df, spectrum


def run_condition(files, cond_name, n_jobs=-1):
    print(f'\n=== {cond_name} ({len(files)} files) ===')
    print(f'Start: {time.strftime("%H:%M:%S")}')
    results      = Parallel(n_jobs=n_jobs)(delayed(process_file)(f) for f in files)
    dfs, spectra = zip(*results)
    df           = pd.concat(dfs, ignore_index=True)
    df['cond']   = cond_name
    return df, list(spectra)


print('process_file() and run_condition() ready.')

In [ ]:
%%time
dfs_all     = {}
spectra_all = {}

for cond_name, files in file_sets.items():
    if not files:
        print(f'[skip] {cond_name}')
        continue
    df, spectra = run_condition(files, cond_name)
    dfs_all[cond_name]     = df
    spectra_all[cond_name] = spectra

all_blocks_df = pd.concat(dfs_all.values(), ignore_index=True)
all_blocks_df.to_csv(
    output_root / f'{DATE_STR}_long_{n}p_continuous_all_blocks.csv',
    index=False
)
print(f'\nTotal rows: {len(all_blocks_df)}')

In [ ]:
# ==============================================================================
# Band integration and stats-ready CSV export
# ==============================================================================

all_blocks_df['freq'] = pd.to_numeric(all_blocks_df['freq'], errors='coerce')
electrodes = [c for c in all_blocks_df.columns
              if c not in ['freq', 'participant', 'cond']]

results = {band: [] for band in OUR_BANDS}

for (participant, cond), group in all_blocks_df.groupby(['participant', 'cond']):
    for band_name, (fmin, fmax) in OUR_BANDS.items():
        filtered = group[
            (group['freq'] > fmin) & (group['freq'] <= fmax)
        ].sort_values('freq')
        freqs = filtered['freq'].to_numpy()
        data  = filtered[electrodes].to_numpy()
        integrated = simpson(data, freqs, axis=0) if len(freqs) > 1 else data.mean(axis=0)
        results[band_name].append({
            'participant': participant,
            'cond':        cond,
            **dict(zip(electrodes, integrated * 1e12))  # V² -> µV²
        })

dfs_bands = {band: pd.DataFrame(rows) for band, rows in results.items()}

# stats-ready: long format, log10 transform
for band, df in dfs_bands.items():
    df_long = df.melt(
        id_vars=['participant', 'cond'],
        value_vars=electrodes,
        var_name='electrode', value_name='power'
    )
    df_long['power'] = np.log10(df_long['power'] + 1e-20)
    out = output_root / f'{DATE_STR}_{n}p_continuous_{band}_stats.csv'
    df_long.to_csv(out, index=False)
    print(f'Saved -> {out.name}')

## Grand average spectra and plots

Flow paper: High Flow vs Low Flow.
Load paper: Simple (oddball runs pooled) vs Complex (UT runs pooled).

In [ ]:
# ==============================================================================
# Grand average spectra per paper
# ==============================================================================

def spectra_to_stack(spectra_list):
    arrays = [s.get_data() for s in spectra_list]
    return np.stack(arrays, axis=0)


info  = spectra_all[list(spectra_all.keys())[0]][0].info.copy()
freqs = spectra_all[list(spectra_all.keys())[0]][0].freqs

if PAPER == 'flow':
    # high flow vs low flow
    plot_conds = {
        'High Flow': spectra_all.get('UT_highflow', []),
        'Low Flow':  spectra_all.get('UT_lowflow',  []),
    }
    my_pal = {'High Flow': '#E74C3C', 'Low Flow': '#2ECC71'}
else:  # load
    # pool oddball runs -> simple, pool UT runs -> complex
    plot_conds = {
        'Simple':  (spectra_all.get('oddball_1', []) + spectra_all.get('oddball_2', [])),
        'Complex': (spectra_all.get('UT_highflow', []) + spectra_all.get('UT_lowflow', [])),
    }
    my_pal = {'Simple': '#3498DB', 'Complex': '#F39C12'}

avg_spectra = {}
for label, spectra_list in plot_conds.items():
    if not spectra_list:
        print(f'[skip] {label} — no spectra')
        continue
    stack = spectra_to_stack(spectra_list)
    avg_spectra[label] = mne.time_frequency.SpectrumArray(
        stack.mean(axis=0), info=info, freqs=freqs
    )
    print(f'Grand avg [{label}]: {stack.shape[0]} spectra averaged')

In [ ]:
# PSD line plot (all channels averaged)

bands_plot  = {'theta': (3.5, 8), 'alpha': (8, 13), 'low_beta': (13, 20), 'high_beta': (20, 30)}
band_colors = {'theta': '#A9CCE3', 'alpha': '#F9E79F', 'low_beta': '#D5A6BD', 'high_beta': '#C39BD3'}
band_labels = {'theta': r'$\theta$', 'alpha': r'$\alpha$',
               'low_beta': r'low-$\beta$', 'high_beta': r'high-$\beta$'}

fig, ax = plt.subplots(figsize=(10, 5))

for band, (fmin, fmax) in bands_plot.items():
    ax.axvspan(fmin, fmax, color=band_colors[band], alpha=0.3)
    ax.text((fmin + fmax) / 2, 0, band_labels[band],
            ha='center', va='bottom', fontsize=13, fontweight='bold',
            transform=ax.get_xaxis_transform())

for cond, spec in avg_spectra.items():
    data     = spec.get_data() * 1e12  # V²/Hz -> µV²/Hz
    mean_psd = 10 * np.log10(data.mean(axis=0))
    ax.plot(freqs, mean_psd, color=my_pal[cond], label=cond, lw=2)

ax.set_xlim(1, 40)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB, µV²/Hz)')
ax.legend()
ax.set_title(f'Continuous PSD — {PAPER} {EXPERIMENT} ({n}p)')
plt.tight_layout()
plt.savefig(img_root / f'{DATE_STR}_{n}p_{PAPER}_{EXPERIMENT}_continuous_psd_dB.svg',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Topomaps per band — per-band scale and global scale

def compute_band_vlim(spectra_dict, band_range, conditions=None):
    if conditions is None:
        conditions = list(spectra_dict.keys())
    fmin, fmax = band_range
    all_vals   = []
    for cond in conditions:
        spec = spectra_dict[cond]
        idx  = (spec.freqs >= fmin) & (spec.freqs <= fmax)
        all_vals.append(spec.get_data()[:, idx].mean(axis=1) * 1e12)
    return float(np.concatenate(all_vals).min()), float(np.concatenate(all_vals).max())


def plot_topo_grid(avg_spectra, vlim_dict, fname_suffix):
    plot_labels = list(avg_spectra.keys())
    n_rows = len(plot_labels)
    n_cols = len(OUR_BANDS)

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4.5 * n_cols, 4.5 * n_rows), squeeze=False)

    for i, cond in enumerate(plot_labels):
        for j, (band_name, band_range) in enumerate(OUR_BANDS.items()):
            ax = axes[i, j]
            avg_spectra[cond].plot_topomap(
                bands={band_name: band_range},
                vlim=vlim_dict[band_name],
                dB=False, axes=ax, show=False
            )
            if i == 0:
                ax.set_title(band_name.replace('_', ' ').capitalize(), fontsize=13)
            if j == 0:
                ax.set_ylabel(cond, fontsize=12, rotation=0, labelpad=60, va='center')

    plt.suptitle(
        f'Continuous PSD topomaps — {PAPER} {EXPERIMENT} ({n}p)\n{fname_suffix}',
        fontsize=14
    )
    plt.tight_layout()
    fname = img_root / f'{DATE_STR}_{n}p_{PAPER}_{EXPERIMENT}_continuous_topo_allbands_{fname_suffix}.svg'
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    print(f'Saved -> {fname.name}')
    plt.show()


plot_labels = list(avg_spectra.keys())

# per-band scale — each band independently normalized
vlims_perband = {band: compute_band_vlim(avg_spectra, rng, plot_labels)
                 for band, rng in OUR_BANDS.items()}
plot_topo_grid(avg_spectra, vlims_perband, 'perband_scale')

# global scale — single scale across all bands
all_vals = []
for band, rng in OUR_BANDS.items():
    fmin, fmax = rng
    for cond, spec in avg_spectra.items():
        idx = (spec.freqs >= fmin) & (spec.freqs <= fmax)
        all_vals.append(spec.get_data()[:, idx].mean(axis=1) * 1e12)
global_vmin = float(np.concatenate(all_vals).min())
global_vmax = float(np.concatenate(all_vals).max())
vlims_global = {band: (global_vmin, global_vmax) for band in OUR_BANDS}
plot_topo_grid(avg_spectra, vlims_global, 'global_scale')